# Chapter 2 - Lab 4: Shipment tracking with the ReAct paradigm (LlamaIndex)

**ReAct** = **Re**asoning + **Act**ing. The agent alternates:
*Thought* (what do I need?) -> *Action* (call a tool) -> *Observation* (read the result) -> ... -> *Answer*.

*Finance version*: fetch Apple's latest price with a Yahoo Finance tool.
*Customer-care version*: fetch the latest courier scan of a NovaMart order with a shipment-tracking tool.

We use LlamaIndex's built-in `ReActAgent` with `gpt-4o`.

In [ ]:
%pip install -q -U llama-index llama-index-llms-openai

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## Define the tool (mock courier tracking API)

In [ ]:
from datetime import datetime, timedelta
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.tools import FunctionTool
from llama_index.llms.openai import OpenAI

NOW = datetime.now()
SHIPMENTS = {
    "ORD-78120": {"courier": "SwiftShip", "awb": "SS884213907IN", "scans": [
        {"time": NOW - timedelta(days=9), "location": "Mumbai fulfilment centre", "event": "Picked up"},
        {"time": NOW - timedelta(days=7), "location": "Delhi (Okhla) hub", "event": "Arrived at hub"},
        {"time": NOW - timedelta(hours=5), "location": "Delhi (Okhla) hub", "event": "Held: hub backlog"},
    ], "eta": (NOW + timedelta(days=2)).strftime("%Y-%m-%d")},
}

def get_latest_scan(order_id: str) -> dict:
    """Fetch the latest courier scan (time, location, event) and the ETA for a NovaMart order."""
    shipment = SHIPMENTS.get(order_id.strip().upper())
    if shipment is None:
        return {"error": f"No shipment found for order {order_id}"}
    last = shipment["scans"][-1]
    return {"order_id": order_id, "courier": shipment["courier"], "awb": shipment["awb"],
            "datetime": last["time"].strftime("%Y-%m-%d %H:%M"), "location": last["location"],
            "event": last["event"], "eta": shipment["eta"]}

tools = [FunctionTool.from_defaults(get_latest_scan)]

## Specify and run the agent

In [ ]:
tracking_agent = ReActAgent(llm=OpenAI(model="gpt-4o"), tools=tools, timeout=120, verbose=False)

ret_1 = await tracking_agent.run("Where is my order ORD-78120 right now?")
print(ret_1.response.blocks[0].text)

## Stream the Thought / Action / Observation trace

In [ ]:
from llama_index.core.agent.workflow import AgentStream

handler = tracking_agent.run("Where is my order ORD-78120 right now?")
async for ev in handler.stream_events():
    if isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)
response = await handler

Every step is visible: *Thought: I need the latest scan* -> *Action: get_latest_scan* ->
*Observation: held at Delhi hub* -> *Answer*. For a supervisor or auditor, this trace explains **why**
the agent told the customer what it did.

## With memory: a follow-up question answered without a second tool call

In [ ]:
from llama_index.core.workflow import Context

tracking_agent = ReActAgent(llm=OpenAI(model="gpt-4o"), tools=tools, timeout=120, verbose=False)
ctx = Context(tracking_agent)

handler_1 = tracking_agent.run("Where is my order ORD-78120 right now?", ctx=ctx)
async for ev in handler_1.stream_events():
    if isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)
await handler_1

print("\n" + "=" * 60)
handler_2 = tracking_agent.run("Which courier has it, and what is the airway bill number?", ctx=ctx)
async for ev in handler_2.stream_events():
    if isinstance(ev, AgentStream):
        print(ev.delta, end="", flush=True)
await handler_2

In the second question the agent only *thinks*: the courier and AWB are already in its context, so it
answers without calling the tool again. That is cheaper and faster.

## Discussion
1. Why is a visible reasoning trace valuable when an agent talks to customers?
2. When should the agent call the tool again instead of trusting memory (hint: status changes every few hours)?